# 01 — Chunk smart: how you cut decides what your RAG can answer

Companion notebook to blog post **01 (Chunk smart)**. Watch the same handbook, retriever,
LLM and question give a RIGHT or WRONG answer depending only on where the scissors fell.

The index-card metaphor (post 00's open book doesn't fit on the desk):
1. The book **must become cards** (chunking is unavoidable)
2. Too big **blurs** — dilution
3. Too small **forgets** — orphans (rule separated from exception)
4. Cut at **natural seams**, **overlap** the edges — then **measure**

Needs an OpenAI API key for the orphan LLM demo only; everything else is local + free.

In [ ]:
%pip install -q fastembed numpy openai

In [ ]:
import os

def load_key():
    try:                                      # Colab: add OPENAI_API_KEY in the Secrets
        from google.colab import userdata     # sidebar (key icon) and enable notebook access
        return userdata.get("OPENAI_API_KEY")
    except Exception:                         # local Jupyter fallback
        import getpass
        return os.environ.get("OPENAI_API_KEY") or getpass.getpass("OpenAI API key: ")

os.environ["OPENAI_API_KEY"] = load_key()

## The handbook grows up — three multi-paragraph pages

Note the last boarding paragraph: a **rule** (refunds in 5 days) and its **exception**
(holidays non-refundable). The whole post hinges on those two sentences.

In [ ]:
PAGES = {
"boarding": """Boarding at Sunnyvale Pet Care Center is available seven days a week. The boarding facility closes at 7 pm on weekdays and 5 pm on weekends. All boarding guests must arrive at least one hour before closing time.

Dogs staying longer than three nights receive a complimentary bath before pickup. Blankets and toys from home are welcome and encouraged for comfort.

Refunds for cancelled boarding are issued within 5 business days. Bookings made for public holidays are non-refundable.""",

"grooming": """Grooming appointments must be booked at least 48 hours in advance. Walk-in grooming is not available at any location.

Our full grooming package includes a bath, haircut, nail trim, and ear cleaning. The full package takes about two hours for most breeds.

Refunds for cancelled grooming appointments are issued within 10 business days. A no-show fee of 25 dollars applies if you miss an appointment without cancelling.""",

"daycare": """Daycare drop off starts at 6:30 am and the last pickup is at 8 pm. A late pickup fee of 15 dollars applies for every 30 minutes after closing.

All daycare pets must have up to date rabies vaccination records on file. New guests must pass a temperament assessment before their first full day.

Daycare payments are charged monthly. Unused daycare days do not roll over to the next month.""",
}

In [ ]:
import numpy as np
from fastembed import TextEmbedding

emb_model = TextEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")

def cosine(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

def embed(texts):
    return list(emb_model.embed(texts))

## Failure mode 1 — Too big: dilution

Same fact, same query — watch the score erode as unrelated topics are stapled onto the
document. One embedding = one position for the WHOLE chunk; every extra topic drags it.

In [ ]:
q = "What time does boarding close on weekends?"
qe = embed([q])[0]

grow = [
    PAGES["boarding"],
    PAGES["boarding"] + "\n\n" + PAGES["grooming"],
    PAGES["boarding"] + "\n\n" + PAGES["grooming"] + "\n\n" + PAGES["daycare"],
]
labels = ["boarding page alone (3 paras)",
          "boarding + grooming stapled (6 paras)",
          "whole handbook as ONE doc (9 paras)"]
for emb, label in zip(embed(grow), labels):
    print(f"  {cosine(qe, emb):.3f}  {label}")

## Failure mode 2 — Too small: orphans

Cut one card per sentence. For the holiday-refund question, the RULE outranks the
EXCEPTION (the query says "refund"/"cancel"; the exception sentence says neither).
Top-1 retrieval hands the LLM the rule and never shows it the exception.

In [ ]:
import re

def sentences(text):
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text.replace("\n\n", " ")) if s.strip()]

sent_chunks = [s for page in PAGES.values() for s in sentences(page)]
para_chunks = [p for page in PAGES.values() for p in page.split("\n\n")]
page_chunks = list(PAGES.values())
sent_embs, para_embs = embed(sent_chunks), embed(para_chunks)

qq = "I booked boarding for a public holiday and want to cancel. Will I get my money back?"
qqe = embed([qq])[0]

print("sentence-chunk top-3:")
for s, i in sorted(((cosine(qqe, e), i) for i, e in enumerate(sent_embs)), reverse=True)[:3]:
    print(f"  {s:.3f}  {sent_chunks[i]}")

print("\nparagraph-chunk top-1:")
for s, i in sorted(((cosine(qqe, e), i) for i, e in enumerate(para_embs)), reverse=True)[:1]:
    print(f"  {s:.3f}  {para_chunks[i]}")

Now feed each top-1 context to the same LLM. Expect the answer to FLIP —
the sentence card lies by omission (true answer: holidays are non-refundable).

In [ ]:
from openai import OpenAI

client = OpenAI()
PROMPT = """Answer the question using ONLY the context below.

Context:
{context}

Question: {question}
Answer:"""

def ask(context, question):
    r = client.chat.completions.create(
        model="gpt-5.4-mini", temperature=0,
        messages=[{"role": "user", "content": PROMPT.format(context=context, question=question)}],
    )
    return r.choices[0].message.content.strip()

top_sent = max(((cosine(qqe, e), i) for i, e in enumerate(sent_embs)))[1]
top_para = max(((cosine(qqe, e), i) for i, e in enumerate(para_embs)))[1]

print("with the sentence chunk :", ask(sent_chunks[top_sent], qq))
print("with the paragraph chunk:", ask(para_chunks[top_para], qq))

## Chunking from scratch — fixed size + overlap

The simplest chunker cuts every N characters — and beheads sentences mid-word
(`All board / ing guests`). Overlap re-includes each chunk's tail in the next one,
so boundary sentences survive whole somewhere.

In [ ]:
def chunk(text, size=150, overlap=0):
    text = " ".join(text.split())
    out, start = [], 0
    while start < len(text):
        out.append(text[start:start + size])
        start += size - overlap
    return out

print("no overlap:")
for c in chunk(PAGES["boarding"], 150, 0)[:2]:
    print(f"  [{c}]")

print("\noverlap=50, chunk 2 (the severed closing-time sentence, now whole):")
print(f"  [{chunk(PAGES['boarding'], 150, 50)[1]}]")

## Measure, don't guess — the scoreboard

Five strategies, eight golden questions. hit@1 = gold answer inside the top-1 chunk;
avg-context-chars = how much text that chunk drags into the prompt.

In [ ]:
GOLDEN = [
    ("What time does boarding close on weekends?", "5 pm"),
    ("Do long boarding stays include a free bath?", "complimentary bath"),
    ("Can I get a refund for a holiday boarding booking?", "non-refundable"),
    ("How far ahead must grooming be booked?", "48 hours"),
    ("What does the full grooming package include?", "nail trim"),
    ("What happens if I miss a grooming appointment without cancelling?", "25 dollars"),
    ("What is the late fee at daycare?", "15 dollars"),
    ("Do unused daycare days carry over?", "do not roll over"),
]

strategies = {
    "whole pages": page_chunks,
    "paragraphs": para_chunks,
    "sentences": sent_chunks,
    "fixed 150ch, no overlap": [c for p in PAGES.values() for c in chunk(p, 150, 0)],
    "fixed 150ch, overlap 50": [c for p in PAGES.values() for c in chunk(p, 150, 50)],
}

for label, chunks in strategies.items():
    embs = embed(chunks)
    hits, total_chars = 0, 0
    for gq, gold in GOLDEN:
        ge = embed([gq])[0]
        score, i = max(((cosine(ge, e), i) for i, e in enumerate(embs)))
        total_chars += len(chunks[i])
        hits += gold in chunks[i]
    print(f"  {label:<24} chunks={len(chunks):<3} hit@1={hits}/8  avg-context-chars={total_chars // 8}")

## PRODUCTION — sentence-aware splitting

LlamaIndex's `SentenceSplitter`: fixed-size in tokens, overlap included, refuses to cut
mid-sentence — seams + size control at once. 512/128 is what a measured sweep picked on
the course's full 1,421-doc corpus (1024→512 improved every metric; 256 broke precision
and faithfulness — the orphan failure at scale). Your corpus has different seams:
sweep, measure, pick.

In [ ]:
%pip install -q llama-index-core

In [ ]:
from llama_index.core import Document
from llama_index.core.node_parser import SentenceSplitter

docs = [Document(text=t, metadata={"page": name}) for name, t in PAGES.items()]

splitter = SentenceSplitter(chunk_size=64, chunk_overlap=16)   # tiny sizes for our tiny pages
nodes = splitter.get_nodes_from_documents(docs)

print(f"{len(nodes)} chunks; none cut mid-sentence:\n")
for n in nodes[:3]:
    print(f"  [{n.get_content()[:110]}...]" if len(n.get_content()) > 110 else f"  [{n.get_content()}]")

## Recap — the index cards

1. **The book must become cards** → chunking is unavoidable
2. **Too big blurs** → dilution: 0.520 → 0.451 as topics pile in; 435 vs 141 context chars
3. **Too small forgets** → orphans: "Yes" to a non-refundable booking
4. **Seams + overlap + a scoreboard** → paragraphs won here (8/8 at 141 chars); the
   course sweep picked 512/128 at scale

**Next: hybrid search** — notebooks 2a (BM25), 2b (vector search), 2c (fusion).